# AutoCloud-Agent — Demo

**Hierarchical Multi-Agent RL for Cloud Resource Management**

This notebook walks through all components:
1. SimPy Cloud Simulation (live visualization)
2. RL Agent Evaluation vs 7 Baselines
3. AutoResearch — LLM-guided reward weight tuning (Karpathy-style)

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, clear_output
import time

from configs.default_config import DEFAULT_CONFIG
from environment.cloud_env import CloudEnv
from environment.node import NodeState
from training.baselines import ThresholdReactive, ThresholdPredictive, StaticN, \
    SingleAgentPPO, KubernetesHPA, PIController, ARIMAPredictive
from training.ippo_trainer import IPPOTrainer
from evaluation.evaluator import Evaluator

print('✓ Imports OK')

---
## Part 1 — Live Simulation Visualization

Watch the cloud cluster evolve in real time: nodes scale up/down, jobs queue and dispatch, P95 latency tracked.

In [ ]:
def render_cluster(ax_nodes, ax_metrics, step, metrics, node_history, sla_history, cost_history):
    """Render current cluster state as a visual grid + time-series."""
    ax_nodes.clear()
    ax_metrics.clear()

    # --- Node grid ---
    state_colors = {
        NodeState.BOOTING:    '#f39c12',
        NodeState.ACTIVE:     '#27ae60',
        NodeState.DRAINING:   '#e74c3c',
        NodeState.TERMINATED: '#95a5a6',
    }
    state_labels = {NodeState.BOOTING: 'Boot', NodeState.ACTIVE: 'Active',
                    NodeState.DRAINING: 'Drain', NodeState.TERMINATED: 'Off'}

    n_cols = 5
    for i in range(20):
        row, col = i // n_cols, i % n_cols
        if i < len(node_history):
            state, cpu = node_history[i]
            color = state_colors[state]
            ax_nodes.add_patch(mpatches.FancyBboxPatch(
                (col * 1.2, -row * 1.2), 1.0, 1.0,
                boxstyle='round,pad=0.05', facecolor=color, edgecolor='white', linewidth=2
            ))
            if state == NodeState.ACTIVE:
                # CPU utilization bar inside the box
                ax_nodes.add_patch(mpatches.FancyBboxPatch(
                    (col * 1.2 + 0.05, -row * 1.2 + 0.05), 0.9 * cpu, 0.3,
                    boxstyle='round,pad=0.02', facecolor='white', alpha=0.7
                ))
            ax_nodes.text(col * 1.2 + 0.5, -row * 1.2 + 0.55,
                          state_labels[state], ha='center', va='center',
                          fontsize=7, fontweight='bold', color='white')
        else:
            ax_nodes.add_patch(mpatches.FancyBboxPatch(
                (col * 1.2, -row * 1.2), 1.0, 1.0,
                boxstyle='round,pad=0.05', facecolor='#ecf0f1', edgecolor='#bdc3c7'
            ))

    sla_ok = metrics['p95_latency'] < DEFAULT_CONFIG.reward.sla_latency_ms or metrics['p95_latency'] == 0
    sla_color = '#27ae60' if sla_ok else '#e74c3c'

    ax_nodes.set_xlim(-0.2, 6.2)
    ax_nodes.set_ylim(-4.5, 1.2)
    ax_nodes.axis('off')
    ax_nodes.set_title(
        f"Step {step:3d} | Active: {metrics['n_active']}  Boot: {metrics['n_booting']}  Drain: {metrics['n_draining']}\n"
        f"CPU: {metrics['mean_cpu_util']:.0%}  Queue: {metrics['queue_len']}  "
        f"P95: {metrics['p95_latency']:.0f}s  Cost: ${metrics['total_cost']:.3f}",
        fontsize=10, color=sla_color
    )

    # Legend
    patches = [mpatches.Patch(color=c, label=l)
               for l, c in [('Booting','#f39c12'),('Active','#27ae60'),
                              ('Draining','#e74c3c'),('Off','#95a5a6')]]
    ax_nodes.legend(handles=patches, loc='lower right', fontsize=7)

    # --- Metrics time-series ---
    steps = range(len(sla_history))
    ax_metrics.plot(steps, sla_history, 'g-', linewidth=1.5, label='SLA Met')
    ax_metrics.axhline(y=1.0, color='g', linestyle='--', alpha=0.3)
    ax2 = ax_metrics.twinx()
    ax2.plot(steps, cost_history, 'b-', linewidth=1, alpha=0.6, label='Cost/step')
    ax_metrics.set_ylim(-0.1, 1.3)
    ax_metrics.set_ylabel('SLA Rate', color='g')
    ax2.set_ylabel('Step Cost ($)', color='b')
    ax_metrics.set_xlabel('Step')
    ax_metrics.set_title('Performance Over Time')
    ax_metrics.grid(alpha=0.3)


def run_visual_episode(policy, policy_name='AutoCloud-Agent', n_steps=60, delay=0.15):
    """Run one episode with live cluster visualization."""
    env = CloudEnv(config=DEFAULT_CONFIG, seed=42)
    obs, _ = env.reset()
    if hasattr(policy, 'reset'):
        policy.reset()

    sla_history, cost_history = [], []

    fig, (ax_nodes, ax_metrics) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f'AutoCloud-Agent Demo — Policy: {policy_name}', fontsize=13, fontweight='bold')

    for step in range(n_steps):
        if hasattr(policy, 'select_action'):
            action = policy.select_action(obs, env)
        else:
            a_so, _, _ = policy.so_agent.act(obs)
            mask = env.get_active_mask()
            a_con, _, _ = policy.con_agent.act(obs, mask)
            a_sch, _, _ = policy.sch_agent.act(obs)
            action = {'scaleout': int(a_so), 'consolidation': a_con, 'scheduling': a_sch}

        obs, _, terminated, truncated, info = env.step(action)
        m = info['metrics']

        sla_ok = float(m['p95_latency'] < DEFAULT_CONFIG.reward.sla_latency_ms or m['p95_latency'] == 0)
        sla_history.append(sla_ok)
        cost_history.append(m['step_cost'])

        node_history = [(n.state, n.cpu_util) for n in env.sim.nodes if n.state != NodeState.TERMINATED]

        render_cluster(ax_nodes, ax_metrics, step + 1, m, node_history, sla_history, cost_history)
        clear_output(wait=True)
        display(fig)
        time.sleep(delay)

        if terminated or truncated:
            break

    plt.close(fig)
    sla_rate = np.mean(sla_history)
    print(f'\n Episode complete | SLA rate: {sla_rate:.1%} | Total cost: ${m["total_cost"]:.3f}')
    return sla_history, cost_history

In [ ]:
# Load trained AutoCloud-Agent
CHECKPOINT_DIR = '../checkpoints'

trainer = IPPOTrainer(config=DEFAULT_CONFIG, seed=0, device='cpu', verbose=False)
import os
if os.path.exists(os.path.join(CHECKPOINT_DIR, 'so_actor_final.pt')):
    trainer.load(CHECKPOINT_DIR, tag='final')
    print('✓ Checkpoint loaded')
else:
    print('⚠ No checkpoint found — using random policy (run train_rl_agents.ipynb first)')

# Run visual episode
sla_h, cost_h = run_visual_episode(trainer, policy_name='AutoCloud-Agent (I-PPO)', n_steps=60, delay=0.1)

---
## Part 2 — AutoCloud-Agent vs 7 Baselines

Compare 8 methods across 3 seeds, 10 episodes each.

In [ ]:
evaluator = Evaluator(
    config=DEFAULT_CONFIG,
    checkpoint_dir=CHECKPOINT_DIR,
    n_episodes=5,
    seeds=[0, 1, 2],
    verbose=True,
)

results = evaluator.evaluate_all()
evaluator.print_table(results)

In [ ]:
# Visualization: grouped bar chart
import matplotlib.pyplot as plt
import numpy as np

methods  = list(results.keys())
metrics  = ['sla_rate', 'cost_efficiency', 'mean_cpu_util', 'node_stability']
labels   = ['SLA Rate', 'Cost Efficiency', 'Mean CPU Util', 'Node Stability']
colors   = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6']

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
fig.suptitle('AutoCloud-Agent vs Baselines', fontsize=14, fontweight='bold')

for ax, metric, label, color in zip(axes, metrics, labels, colors):
    means = [results[m]['mean'].get(metric, 0) for m in methods]
    stds  = [results[m]['std'].get(metric, 0)  for m in methods]

    bars = ax.bar(range(len(methods)), means, yerr=stds, capsize=4,
                  color=[color if m == 'AutoCloud-Agent' else '#bdc3c7' for m in methods],
                  edgecolor='white', linewidth=0.8)

    ax.set_xticks(range(len(methods)))
    ax.set_xticklabels([m.replace('Threshold', 'Thr-').replace('Predictive', 'Pred')
                        for m in methods], rotation=45, ha='right', fontsize=7)
    ax.set_title(label, fontweight='bold')
    ax.set_ylim(0, 1.15)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    # Highlight best bar
    best_idx = int(np.argmax(means))
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(2)

plt.tight_layout()
plt.savefig('../results_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: results_comparison.png')

---
## Part 3 — AutoResearch: LLM-Guided Reward Tuning

**Karpathy-style**: the LLM reads `experiment.py` (the current config code),
proposes a rewrite, runs a fast trial, and keeps or discards the change.

This is the same loop Karpathy's autoresearch uses for GPT training — adapted here
for RL reward weight search.

In [ ]:
# Show the research program
with open('../program.md') as f:
    print(f.read())

In [ ]:
# Show the current experiment.py (the file the LLM will modify)
with open('../experiment.py') as f:
    print(f.read())

In [ ]:
import os
from autoresearch.engine import AutoResearchEngine

# ── Choose ONE provider and set the key ───────────────────────────────────
#
# Option A: Groq  (FREE — fastest, recommended)
#   Sign up at https://console.groq.com  (no credit card needed)
#   pip install groq
os.environ['GROQ_API_KEY'] = 'YOUR_GROQ_KEY_HERE'   # ← replace this
LLM_PROVIDER = 'groq'
LLM_MODEL    = 'llama-3.1-8b-instant'
#
# Option B: Ollama  (FREE — fully local, no internet needed)
#   Install: curl -fsSL https://ollama.com/install.sh | sh
#   Then:    ollama pull llama3.2:3b  &&  ollama serve
# LLM_PROVIDER = 'ollama'
# LLM_MODEL    = 'llama3.2:3b'
#
# Option C: Google Gemini  (FREE tier — 60 req/min)
#   Sign up at https://aistudio.google.com  (no credit card)
#   pip install google-generativeai
# os.environ['GEMINI_API_KEY'] = 'YOUR_GEMINI_KEY'
# LLM_PROVIDER = 'gemini'
# LLM_MODEL    = 'gemini-1.5-flash-8b'
# ─────────────────────────────────────────────────────────────────────────

engine = AutoResearchEngine(
    n_iterations=4,       # baseline + 3 experiments
    total_steps=2000,     # ~2 min per trial on CPU
    seed=0,
    trial_timeout=300,    # 5-min hard timeout per trial
    verbose=True,
    llm_provider=LLM_PROVIDER,
    llm_model=LLM_MODEL,
)

print(f'Starting AutoResearch — provider={LLM_PROVIDER}, model={LLM_MODEL}')
print('Loop: LLM reads experiment.py → proposes change → trial → keep/discard\n')

best_score, best_code = engine.run()

In [ ]:
# Plot AutoResearch progress
import pandas as pd
import matplotlib.pyplot as plt

df = pd.DataFrame(engine.history)
if len(df) == 0:
    print('No trials completed (check ANTHROPIC_API_KEY)')
else:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('AutoResearch — Karpathy-Style LLM Reward Tuning', fontweight='bold')

    # Score over iterations
    kept_mask    = df['kept'].values
    discarded    = ~kept_mask
    ax1.plot(df['iter'], df['score'], 'o-', color='#3498db', linewidth=2, label='Score')
    ax1.scatter(df['iter'][kept_mask],    df['score'][kept_mask],    color='#27ae60', s=100, zorder=5, label='KEEP')
    ax1.scatter(df['iter'][discarded], df['score'][discarded], color='#e74c3c', s=100, marker='x', zorder=5, label='DISCARD')
    ax1.axhline(y=df['score'].max(), color='gold', linestyle='--', alpha=0.7, label=f'Best={df["score"].max():.4f}')
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Score = SLA − 0.1×Cost')
    ax1.set_title('Score per Iteration')
    ax1.legend()
    ax1.grid(alpha=0.3)

    # SLA vs Cost tradeoff
    ax2.scatter(df['cost'], df['sla'], c=['#27ae60' if k else '#e74c3c' for k in kept_mask], s=80)
    for _, row in df.iterrows():
        ax2.annotate(f"iter {int(row['iter'])}", (row['cost'], row['sla']),
                     textcoords='offset points', xytext=(5, 5), fontsize=8)
    ax2.set_xlabel('Mean Cost ($)')
    ax2.set_ylabel('SLA Rate')
    ax2.set_title('SLA vs Cost Tradeoff')
    ax2.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig('../autoresearch_progress.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(f'\nBest score: {best_score:.4f}')
    print(f'Results log: autoresearch/results.tsv')

In [ ]:
# Show the final best experiment.py found by AutoResearch
print('=== Best experiment.py found by AutoResearch ===')
with open('../experiment.py') as f:
    print(f.read())

---
## Summary

| Component | What was shown |
|-----------|---------------|
| **SimPy Simulation** | Live cluster: nodes boot/drain, jobs queue, P95 latency tracked |
| **I-PPO Agents** | 3 agents (ScaleOut, Consolidation, Scheduling) + SafetyCoordinator |
| **Baselines** | 8 methods: KubernetesHPA, PIController, ARIMA, ThresholdReactive, ThresholdPredictive, StaticN, SingleAgentPPO |
| **AutoResearch** | Karpathy-style: LLM reads experiment.py → proposes code change → trial → keep/discard |